# U1-01 · Python, NumPy, pandas y Matplotlib con datos de la economía

*Machine Learning aplicado usando Python (IFCD093PO) · Unidad 1, apartados 1.1 a 1.3*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador, sin instalar nada. El archivo `macrodata.csv` está junto a este cuaderno.

> Datos: *macrodata*, incluidos en statsmodels; Reserva Federal de San Luis (FRED) y Oficina de Estadísticas Laborales de EE. UU. Dominio público.

## 1. Python sin bibliotecas

In [ ]:
import csv

with open("macrodata.csv") as f:          # 203 trimestres, 1959-2009
    filas = list(csv.DictReader(f))
print(len(filas), "trimestres; el primero:", filas[0]["year"],
      filas[0]["quarter"], "PIB", filas[0]["realgdp"])

def crecimiento(antes, despues):
    """Variación en porcentaje entre dos valores."""
    return 100 * (despues - antes) / antes

pib = [float(f["realgdp"]) for f in filas]     # comprensión de lista
crec = [crecimiento(a, b) for a, b in zip(pib, pib[1:])]
print(f"crecimiento trimestral medio: {sum(crec) / len(crec):.2f} %")

# recesión técnica: dos trimestres seguidos con el PIB a la baja
recesiones = []
for i in range(1, len(crec)):
    if crec[i - 1] < 0 and crec[i] < 0:
        f = filas[i + 1]                       # crec[i] acaba en i+1
        recesiones.append(f"{f['year']}T{f['quarter']}")
print("trimestres en recesión técnica:", recesiones)

paro = {}                                      # década -> valores
for f in filas:
    decada = int(f["year"]) // 10 * 10
    paro.setdefault(decada, []).append(float(f["unemp"]))
for decada, v in paro.items():
    print(f"años {decada}: paro medio {sum(v) / len(v):.1f} %")

**Prueba.** Cambia la definición de recesión a *tres* trimestres seguidos con el PIB a la baja. ¿Cuántos episodios quedan?

## 2. NumPy: operar con vectores

En el navegador los tiempos son distintos de los de un ordenador con Python instalado.

In [ ]:
import time
import numpy as np

pib = np.loadtxt("macrodata.csv", delimiter=",", skiprows=1,
                 usecols=2)                    # columna realgdp
crec = 100 * (pib[1:] - pib[:-1]) / pib[:-1]   # todo a la vez
print(pib.shape, f"media {crec.mean():.2f} %, mínimo "
      f"{crec.min():.2f} %, máximo {crec.max():.2f} %")
recesion = (crec[:-1] < 0) & (crec[1:] < 0)    # dos caídas seguidas
print("trimestres en recesión técnica:", recesion.sum())

# la misma cuenta repetida 10 000 veces: bucle frente a NumPy
t0 = time.perf_counter()
for _ in range(10_000):
    c = [100 * (b - a) / a for a, b in zip(pib, pib[1:])]
t1 = time.perf_counter()
for _ in range(10_000):
    c = 100 * (pib[1:] - pib[:-1]) / pib[:-1]
t2 = time.perf_counter()
print(f"bucle {t1 - t0:.2f} s, NumPy {t2 - t1:.3f} s: "
      f"{(t1 - t0) / (t2 - t1):.0f} veces más rápido")

## 3. pandas y Matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

d = pd.read_csv("macrodata.csv")
d.index = pd.PeriodIndex.from_fields(year=d["year"],
                                     quarter=d["quarter"], freq="Q")
# crecimiento anual: cada trimestre frente al mismo de un año antes
d["crec_anual"] = 100 * d["realgdp"].pct_change(4)
print(d[["realgdp", "crec_anual", "unemp", "infl"]].tail(3))
decada = d["year"] // 10 * 10
decadas = d.groupby(decada)[["unemp", "infl"]].mean()
print(decadas.round(1))
peor = d["crec_anual"].idxmin()
print(f"peor año: el que acaba en {peor}, PIB "
      f"{d.loc[peor, 'crec_anual']:.1f} %")

fig, eje = plt.subplots(figsize=(4.4, 2.0))
eje.plot(d.index.to_timestamp(), d["unemp"], label="paro (%)")
eje.plot(d.index.to_timestamp(), d["infl"], label="inflación (%)",
         lw=0.8)
for t in d.index[d["crec_anual"] < 0]:        # años de PIB a la baja
    eje.axvspan(t.start_time, t.end_time, color="0.88", lw=0)
eje.legend(); plt.show()  # fig.savefig("macro.pdf")

**Prueba.** Calcula el paro máximo de cada década con `.max()` en lugar de `.mean()`.

## 4. Primer caso: el paro y el crecimiento

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error as eam

d = pd.read_csv("macrodata.csv")
d["g"] = 100 * d["realgdp"].pct_change(4)   # crecimiento anual (%)
d["du"] = d["unemp"].diff(4)          # cambio del paro, en puntos
d = d.dropna()
ent = d[d["year"] < 1990]                    # el pasado: aprende
pru = d[d["year"] >= 1990]                   # el futuro: prueba
X, y, Xp, yp = ent[["g"]], ent["du"], pru[["g"]], pru["du"]
print(f"entrenamiento {len(ent)} trimestres, prueba {len(pru)}")

def regla(g):                # escrita a mano, con números redondos
    return -0.5 * (g - 3)

modelo = LinearRegression().fit(X, y)        # aprende de los datos
b, a = modelo.coef_[0], modelo.intercept_
print(f"aprendido: cambio del paro = {b:.3f} * g {a:+.3f}")
print(f"el paro no cambia si el PIB crece {-a / b:.2f} %")

for nombre, pred in [("referencia: sin cambios", 0 * yp),
                     ("regla a mano", regla(pru["g"])),
                     ("modelo aprendido", modelo.predict(Xp))]:
    print(f"{nombre:<24} error medio {eam(yp, pred):.3f} puntos")

g = np.linspace(-5, 9, 2)
plt.scatter(ent["g"], ent["du"], s=4, label="1960-1989")
plt.scatter(pru["g"], pru["du"], s=4, label="1990-2009")
plt.plot(g, b * g + a, "k", lw=1, label="aprendido")
plt.plot(g, regla(g), "k--", lw=1, label="regla a mano")
plt.xlabel("crecimiento del PIB en un año (%)")
plt.ylabel("cambio del paro (puntos)")
plt.legend(); plt.show()  # plt.savefig("okun.pdf")

---
## Listados de ampliación

Los listados siguientes completan los apartados del libro, en el mismo orden. Los que reutilizan un caso anterior lo importan de los ficheros `.py` que están junto a este cuaderno (por ejemplo, `from h_ventas import X, y`).

### Estructuras de datos  ·  `a2_estructuras.py`

In [ ]:
import csv

with open("macrodata.csv") as f:
    filas = list(csv.DictReader(f))
paro = [float(f["unemp"]) for f in filas]          # lista
anyos = {int(f["year"]) for f in filas}            # conjunto
print("años distintos:", len(anyos), "| primero y último:",
      min(anyos), max(anyos))
print("paro en los 4 primeros trimestres:", paro[:4],
      "| en los 2 últimos:", paro[-2:])

por_decada = {}                                    # diccionario
for f in filas:
    decada = int(f["year"]) // 10 * 10
    por_decada.setdefault(decada, []).append(float(f["unemp"]))
for decada, valores in sorted(por_decada.items()):
    media = sum(valores) / len(valores)
    print(f"{decada}s: paro medio {media:4.1f} % "
          f"({len(valores)} trimestres)")

maximo = max(zip(paro, [(f["year"], f["quarter"]) for f in filas]))
print("paro más alto:", maximo)                    # tupla
i = 0
while paro[i] < 7:                                 # primer >= 7 %
    i += 1
print("primer trimestre con paro de 7 % o más:",
      filas[i]["year"], filas[i]["quarter"])

### Comprensiones, zip y generadores  ·  `a6_comprensiones.py`

In [ ]:
import csv

with open("macrodata.csv") as f:              # sin bibliotecas
    filas = list(csv.DictReader(f))
paro = [float(r["unemp"]) for r in filas]     # lista por comprensión
años = sorted({int(r["year"]) for r in filas})  # sin repetir
print(len(paro), "trimestres de", años[0], "a", años[-1])
media_año = {a: sum(float(r["unemp"]) for r in filas
                    if int(r["year"]) == a) / 4
             for a in años if a < 2009}       # diccionario
peor = max(media_año, key=media_año.get)
print("año con más paro:", peor, round(media_año[peor], 2), "%")
# zip empareja listas; enumerate numera
fecha = [f"{r['year']}T{r['quarter']}" for r in filas]
saltos = [f"{fecha[i + 1]} (+{b - a:.1f})" for i, (a, b)
          in enumerate(zip(paro, paro[1:])) if b - a >= 1]
print("subidas de un punto o más:", ", ".join(saltos))
# un generador no guarda la lista: calcula al pedirle valores
print("trimestres con paro > 7 %:", sum(1 for p in paro if p > 7))

### Funciones, errores y pruebas  ·  `a3_funciones.py`

In [ ]:
def tasa_anual(trimestral):
    """Convierte una tasa trimestral (en %) en anual compuesta."""
    if trimestral <= -100:
        raise ValueError("una tasa no puede bajar del -100 %")
    return 100 * ((1 + trimestral / 100) ** 4 - 1)

def resumen(valores, *, decimales=2):   # argumento con nombre
    n = len(valores)
    media = sum(valores) / n
    var = sum((v - media) ** 2 for v in valores) / (n - 1)
    return round(media, decimales), round(var ** 0.5, decimales)

assert abs(tasa_anual(0) - 0) < 1e-12          # pruebas mínimas
assert abs(tasa_anual(1) - 4.060401) < 1e-6
print("1 % trimestral =", round(tasa_anual(1), 3), "% anual")
print("resumen:", resumen([1.2, 0.8, -0.5, 1.1]),
      resumen([1.2, 0.8, -0.5, 1.1], decimales=4))
try:
    tasa_anual(-150)
except ValueError as e:
    print("error controlado:", e)
cuadrado = lambda x: x ** 2                  # función anónima
print(sorted([-3, 1, -2], key=cuadrado))

### Validar datos y registrar lo que pasa  ·  `a7_validar.py`

In [ ]:
import logging
import pandas as pd

logging.basicConfig(level=logging.INFO,
                    format="%(levelname)s %(message)s")
log = logging.getLogger("tienda")

def validar(t):
    """Comprueba la tabla y devuelve las líneas válidas."""
    faltan = {"Invoice", "Quantity", "Price"} - set(t.columns)
    if faltan:
        raise ValueError(f"faltan columnas: {faltan}")
    malas = (t.Quantity <= 0) | (t.Price <= 0)
    if malas.mean() > 0.10:
        log.warning("%.1f %% de líneas no válidas",
                    100 * malas.mean())
    log.info("%d líneas válidas de %d", (~malas).sum(), len(t))
    return t[~malas]

t = pd.read_csv("tienda_dic2010.csv")
v = validar(t)
assert (v.Quantity > 0).all(), "quedan cantidades no positivas"
try:
    validar(t.drop(columns="Price"))
except ValueError as e:
    log.error("no se puede usar el fichero: %s", e)

### Clases  ·  `a4_clases.py`

In [ ]:
import csv

class Serie:
    """Una serie temporal trimestral con nombre."""
    def __init__(self, nombre, valores):
        self.nombre, self.valores = nombre, list(valores)
    def __len__(self):
        return len(self.valores)
    def __repr__(self):
        return f"Serie({self.nombre!r}, {len(self)} trimestres)"
    def crecimiento(self, retardo=1):
        v = self.valores
        return [100 * (v[i] - v[i - retardo]) / v[i - retardo]
                for i in range(retardo, len(v))]

class SerieSuavizable(Serie):        # herencia
    def media_movil(self, k=4):
        v = self.valores
        return [sum(v[i - k + 1:i + 1]) / k
                for i in range(k - 1, len(v))]

with open("macrodata.csv") as f:
    filas = list(csv.DictReader(f))
pib = SerieSuavizable("PIB real",
                              (float(f["realgdp"]) for f in filas))
print(pib, "| es una Serie:", isinstance(pib, Serie))
anual = pib.crecimiento(4)
print("crecimiento anual, últimos 4:",
      [round(c, 2) for c in anual[-4:]])
print("media móvil, primeros 2:",
      [round(m, 1) for m in pib.media_movil()[:2]])

### Ficheros, carpetas y versiones  ·  `a5_ficheros.py`

In [ ]:
import json
from pathlib import Path
from importlib.metadata import version
import pandas as pd

carpeta = Path("resultados")
carpeta.mkdir(exist_ok=True)               # crea si no existe
d = pd.read_csv("macrodata.csv")
d[["year", "quarter", "unemp"]].to_csv(carpeta / "paro.csv",
                                       index=False)
d.to_json(carpeta / "macro.json", orient="records")
config = {"datos": "macrodata.csv", "filas": len(d),
          "bibliotecas": {b: version(b) for b in
                          ("numpy", "pandas", "scikit-learn")}}
(carpeta / "config.json").write_text(json.dumps(config, indent=1))
for fichero in sorted(carpeta.iterdir()):
    print(f"{fichero.name:<12} {fichero.stat().st_size:>7} bytes")
print(json.loads((carpeta / "config.json").read_text()))

### Matrices y difusión  ·  `b3_matrices.py`

In [ ]:
import numpy as np

datos = np.loadtxt("macrodata.csv", delimiter=",", skiprows=1)
X = datos[:, [9, 10, 12]]           # tipo de interés, paro, inflación
print("forma:", X.shape, "| medias:", X.mean(axis=0).round(2))
# difusión (broadcasting): (203, 3) - (3,) resta a cada fila
Z = (X - X.mean(axis=0)) / X.std(axis=0)
print("tras estandarizar:", Z.mean(axis=0).round(12) + 0,
      Z.std(axis=0))
R = Z.T @ Z / len(Z)                # correlaciones con un producto
print(R.round(2))
print("igual que np.corrcoef:",
      np.allclose(R, np.corrcoef(X, rowvar=False)))
# distancias entre los 4 primeros trimestres: (4,1,3) - (1,4,3)
D = np.sqrt(((Z[:4, None, :] - Z[None, :4, :]) ** 2).sum(axis=2))
print(D.round(2))

### Series temporales con pandas y statsmodels  ·  `b2_co2.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose

co2 = sm.datasets.co2.load_pandas().data["co2"]  # semanal, Mauna Loa
print(co2.size, "semanas,", co2.isna().sum(), "sin dato;",
      co2.index[0].date(), "a", co2.index[-1].date())
mes = co2.resample("MS").mean().interpolate()    # media mensual
print("meses:", mes.size, "| primer y último valor:",
      round(mes.iloc[0], 1), round(mes.iloc[-1], 1))
anual = mes.groupby(mes.index.year).mean()
print("subida media por año (ppm):",
      round(np.diff(anual.values[1:-1]).mean(), 2))
partes = seasonal_decompose(mes, model="additive", period=12)
est = partes.seasonal
ciclo = est.groupby(est.index.month).mean()    # efecto de cada mes
print("ciclo anual: máximo en el mes", ciclo.idxmax(),
      f"({ciclo.max():+.2f}), mínimo en el {ciclo.idxmin()}"
      f" ({ciclo.min():+.2f})")
fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.8),
                           layout="constrained")
a.plot(mes); a.plot(partes.trend); a.set_title("A. CO$_2$ (ppm)")
a.set_xticks(pd.to_datetime(["1960", "1980", "2000"]),
             ["1960", "1980", "2000"])
b.bar(ciclo.index, ciclo.values); b.set_title("B. Ciclo anual")
b.set_xticks(range(1, 13, 2))
plt.show()  # fig.savefig("co2.pdf")

### Datos de panel: agrupar, pivotar y unir  ·  `c2_panel.py`

In [ ]:
import pandas as pd
import statsmodels.api as sm

g = sm.datasets.grunfeld.load_pandas().data   # 11 empresas, 1935-54
g["year"] = g["year"].astype(int)
print(g.shape, "| empresas:", g["firm"].nunique(), "| años:",
      g["year"].min(), "a", g["year"].max())
resumen = (g.groupby("firm")
             .agg(inversion=("invest", "mean"),
                  valor=("value", "mean"))
             .sort_values("inversion", ascending=False).round(1))
print(resumen.head(3))
tabla = g.pivot(index="year", columns="firm", values="invest")
print("tabla año x empresa:", tabla.shape)
crec = tabla.pct_change().mean().sort_values()      # por columna
print("crecimiento medio anual de la inversión, extremos:")
print((100 * crec.iloc[[0, -1]]).round(1).to_string())
total = g.groupby("year")["invest"].sum().rename("total")
g = g.merge(total, left_on="year", right_index=True)  # unir
g["cuota"] = 100 * g["invest"] / g["total"]
gm = g[g.firm == "General Motors"].set_index("year")["cuota"]
print("cuota de General Motors: 1935", round(gm[1935], 1),
      "%, 1954", round(gm[1954], 1), "%")

### Limpiar datos reales  ·  `c5_limpieza.py`

In [ ]:
import pandas as pd

t = pd.read_csv("tienda_dic2010.csv", parse_dates=["InvoiceDate"])
print(t.shape, "| columnas:", ", ".join(t.columns[:4]), "...")
print("valores ausentes por columna:")
print(t.isna().sum()[t.isna().sum() > 0].to_string())
anulada = t["Invoice"].astype(str).str.startswith("C")
print("líneas de facturas anuladas:", anulada.sum(),
      "| cantidades negativas:", (t["Quantity"] < 0).sum(),
      "| precios nulos o negativos:", (t["Price"] <= 0).sum())
print("líneas duplicadas exactas:", t.duplicated().sum())
limpia = (t[~anulada & (t["Quantity"] > 0) & (t["Price"] > 0)]
          .dropna(subset=["Customer ID"]).drop_duplicates())
limpia = limpia.assign(importe=limpia["Quantity"] * limpia["Price"])
print("quedan", len(limpia), "de", len(t), "líneas",
      f"({len(limpia) / len(t):.0%})")
q1, q3 = limpia["importe"].quantile([0.25, 0.75])
atipico = limpia["importe"] > q3 + 3 * (q3 - q1)
med = limpia["importe"].median()
print(f"importe por línea: mediana {med:.2f} £,"
      f" máximo {limpia['importe'].max():.2f} £, muy altos "
      f"{atipico.sum()}")

### Fechas y texto  ·  `c6_fechas_texto.py`

In [ ]:
import pandas as pd

t = pd.read_csv("tienda_dic2010.csv", parse_dates=["InvoiceDate"])
t = t[(t.Quantity > 0) & (t.Price > 0)]
t["importe"] = t.Quantity * t.Price
f = t.InvoiceDate                         # accesor .dt para fechas
dias = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
por_dia = t.groupby(f.dt.dayofweek).importe.sum().div(1000)
print("miles de £:", {dias[i]: round(v, 1)
                      for i, v in por_dia.items()})
por_hora = t.groupby(f.dt.hour).Invoice.nunique()
print(f"facturas de {por_hora.index.min()} a {por_hora.index.max()}"
      f" h; más a las {por_hora.idxmax()} h ({por_hora.max()})")
diario = t.set_index("InvoiceDate").importe.resample("D").sum()
print("días sin ventas en el periodo:", (diario == 0).sum(),
      "de", len(diario))
# accesor .str para texto
d = t.Description.str.strip().str.upper()
print("descripciones con dobles espacios:",
      d.str.contains("  ").sum())
palabras = d.str.split().explode()
print(palabras.value_counts().head(6).to_dict())
color = d.str.extract(r"\b(RED|PINK|BLUE|GREEN|WHITE)\b")[0]
print(color.value_counts().to_dict())

### Tablas dinámicas y varias medidas a la vez  ·  `c9_dinamica.py`

In [ ]:
import pandas as pd

t = pd.read_csv("tienda_dic2010.csv", parse_dates=["InvoiceDate"])
t = t[(t.Quantity > 0) & (t.Price > 0)]
t = t.dropna(subset=["Customer ID"])          # solo con cliente
t["importe"] = t.Quantity * t.Price
t["semana"] = t.InvoiceDate.dt.isocalendar().week
t["zona"] = t.Country.where(t.Country == "United Kingdom",
                            "otros países")
tabla = t.pivot_table(index="semana", columns="zona",
                      values="importe", aggfunc="sum",
                      margins=True, margins_name="total")
print((tabla / 1000).round(1).to_string())
# varias medidas a la vez por cliente
clientes = t.groupby("Customer ID").agg(
    facturas=("Invoice", "nunique"), lineas=("Invoice", "size"),
    importe=("importe", "sum"))
print(clientes.describe().loc[["mean", "50%", "max"]].round(1)
      .to_string())

### Tipos de datos y memoria  ·  `c10_memoria.py`

In [ ]:
import pandas as pd

t = pd.read_csv("tienda_dic2010.csv")
mb = lambda d: d.memory_usage(deep=True).sum() / 1e6   # megabytes
print(f"tal como se lee: {mb(t):.1f} MB")
c = t.copy()
for col in ["StockCode", "Description", "Country"]:
    c[col] = c[col].astype("category")      # valores repetidos
c["Quantity"] = pd.to_numeric(c.Quantity, downcast="integer")
c["Price"] = pd.to_numeric(c.Price, downcast="float")
print(f"con categorías y tipos pequeños: {mb(c):.1f} MB")
print("valores distintos:", {k: t[k].nunique() for k in
                             ["StockCode", "Description",
                              "Country"]})
antes = (t.Quantity * t.Price).sum()
despues = float((c.Quantity * c.Price).sum())
print(f"ventas totales: antes {antes:.2f}, después {despues:.2f}")

### Consultas SQL desde Python  ·  `c8_sql.py`

In [ ]:
import sqlite3
import pandas as pd

t = pd.read_csv("tienda_dic2010.csv")
con = sqlite3.connect(":memory:")       # base de datos en memoria
t.to_sql("lineas", con, index=False)
consulta = """
    SELECT Country AS pais,
           COUNT(DISTINCT Invoice) AS facturas,
           ROUND(SUM(Quantity * Price), 0) AS importe
    FROM lineas
    WHERE Quantity > 0 AND Price > 0
    GROUP BY Country
    ORDER BY importe DESC
    LIMIT 4"""
sql = pd.read_sql(consulta, con)
print(sql.to_string(index=False))
# lo mismo con pandas
v = t[(t.Quantity > 0) & (t.Price > 0)]
pd_ = (v.assign(importe=v.Quantity * v.Price)
        .groupby("Country")
        .agg(facturas=("Invoice", "nunique"),
             importe=("importe", "sum"))
        .round(0).nlargest(4, "importe"))
print("mismo resultado:",
      (pd_.importe.to_numpy() == sql.importe.to_numpy()).all())
con.close()

### Modelos con fórmulas en statsmodels  ·  `c3_formulas.py`

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

g = sm.datasets.grunfeld.load_pandas().data
m1 = smf.ols("invest ~ value + capital", data=g).fit()
m2 = smf.ols("invest ~ value + capital + C(firm)", data=g).fit()
for nombre, m in (("sin empresa", m1), ("con efecto empresa", m2)):
    print(f"{nombre:<19} value {m.params['value']:.4f} "
          f"(IC {m.conf_int().loc['value'].round(4).tolist()}), "
          f"capital {m.params['capital']:.4f}, R2 {m.rsquared:.3f}")
print("¿aporta la empresa? p =",
      f"{m2.compare_f_test(m1)[1]:.1e}")

### Matplotlib, scikit-learn y statsmodels  ·  `c4_api.py`

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

r = sm.datasets.randhie.load_pandas().data     # 20 190 personas
y = (r.pop("mdvis") > 0).astype(int)           # ¿fue al médico?
X_e, X_p, y_e, y_p = train_test_split(r, y, test_size=0.25,
                                      random_state=0, stratify=y)
modelos = {"siempre la clase más frecuente": DummyClassifier(),
           "regresión logística": make_pipeline(
               StandardScaler(), LogisticRegression()),
           "árbol (profundidad 5)": DecisionTreeClassifier(
               max_depth=5, random_state=0),
           "15 vecinos": make_pipeline(
               StandardScaler(), KNeighborsClassifier(15))}
print(f"visitan al médico: {y.mean():.1%} de {len(y)} personas")
for nombre, m in modelos.items():           # la misma interfaz
    m.fit(X_e, y_e)
    print(f"{nombre:<31} acierto {m.score(X_p, y_p):.3f}")

### Gráficos con Matplotlib  ·  `c7_graficos.py`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

t = pd.read_csv("tienda_dic2010.csv", parse_dates=["InvoiceDate"])
t = t[(t.Quantity > 0) & (t.Price > 0)]
imp = t.Quantity * t.Price
fig, (a, b, c) = plt.subplots(1, 3, figsize=(4.4, 1.7),
                              layout="constrained")
a.hist(imp, bins=50)                           # escala lineal
a.set_title("Importe por línea"); a.set_xlabel("£")
b.hist(imp, bins=np.logspace(-1, 4, 50))       # escala logarítmica
b.set_xscale("log"); b.set_title("Escala logarítmica")
b.set_xlabel("£")
pais = t.groupby("Country").Invoice.nunique().nlargest(5)
es = {"United Kingdom": "R. Unido", "Germany": "Alemania",
      "France": "Francia", "EIRE": "Irlanda"}
c.barh([es.get(p, p) for p in pais.index], pais.values)
c.invert_yaxis(); c.set_title("Facturas por país")
plt.show()  # fig.savefig("graficos.pdf")
print("importe: media", round(imp.mean(), 2), "| mediana",
      round(imp.median(), 2), "| percentil 99",
      round(imp.quantile(0.99), 2))
print(pais.to_dict())

### Separar bien los datos  ·  `d2_fuga.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.model_selection import cross_val_score

d = pd.read_csv("macrodata.csv")
X = pd.DataFrame({f"paro_{k}": d["unemp"].shift(k) for k in (1, 2)})
X["pib_1"] = 100 * d["realgdp"].pct_change().shift(1)
y = d["unemp"]                          # paro del trimestre siguiente
datos = pd.concat([X, y], axis=1).dropna()
X, y = datos.iloc[:, :-1], datos["unemp"]
for nombre, m in (("lineal", LinearRegression()),
                  ("bosque", RandomForestRegressor(random_state=0))):
    for cv_nombre, cv in (("al azar", KFold(5, shuffle=True,
                                             random_state=0)),
                          ("en orden", TimeSeriesSplit(5))):
        e = -cross_val_score(m, X, y, cv=cv,
                             scoring="neg_mean_absolute_error")
        print(f"{nombre:<7} partición {cv_nombre:<8}: error medio "
              f"{e.mean():.3f} puntos de paro")

### Datos que no representan el problema  ·  `d7_sesgo.py`

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

r = sm.datasets.randhie.load_pandas().data
y = (r.pop("mdvis") > 0).astype(int)
X_e, X_p, y_e, y_p = train_test_split(r, y, test_size=0.25,
                                      random_state=0, stratify=y)
sano = X_e.disea < X_e.disea.median()    # muestra sesgada: sanos
azar = X_e.sample(sano.sum(), random_state=0).index  # mismo tamaño
for nombre, Xe, ye in [("muestra completa", X_e, y_e),
                       ("mitad al azar", X_e.loc[azar], y_e[azar]),
                       ("solo los más sanos", X_e[sano], y_e[sano])]:
    m = HistGradientBoostingClassifier(random_state=0).fit(Xe, ye)
    p = m.predict_proba(X_p)[:, 1]
    print(f"{nombre:<19} n={len(ye):5d}  AUC prueba "
          f"{roc_auc_score(y_p, p):.3f}")

### Ni muy simple ni muy complejo  ·  `d3_complejidad.py`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.neighbors import KNeighborsRegressor

s = sm.datasets.sunspots.load_pandas().data["SUNACTIVITY"].values
X = np.column_stack([s[i:len(s) - 3 + i] for i in range(3)])
y = s[3:]                         # año siguiente a tres años dados
X_e, y_e, X_p, y_p = X[:220], y[:220], X[220:], y[220:]
ks, ent, pru = range(1, 61), [], []
for k in ks:
    m = KNeighborsRegressor(k).fit(X_e, y_e)
    ent.append(np.mean(np.abs(m.predict(X_e) - y_e)))
    pru.append(np.mean(np.abs(m.predict(X_p) - y_p)))
mejor = int(np.argmin(pru)) + 1
print(f"k = 1: entrenamiento {ent[0]:.1f}, prueba {pru[0]:.1f}")
print(f"k = {mejor}: entrenamiento {ent[mejor - 1]:.1f}, prueba "
      f"{pru[mejor - 1]:.1f} (el mejor en prueba)")
print(f"k = 60: entrenamiento {ent[-1]:.1f}, prueba {pru[-1]:.1f}")
print(f"repetir el último año: prueba "
      f"{np.mean(np.abs(X_p[:, -1] - y_p)):.1f}")
plt.figure(figsize=(4.4, 2.0))
plt.plot(ks, ent, label="entrenamiento")
plt.plot(ks, pru, label="prueba")
plt.xlabel("k (vecinos)"); plt.ylabel("error medio")
plt.legend(); plt.show()  # plt.savefig("complejidad.pdf")

### Un proyecto completo en cinco pasos  ·  `d4_proyecto.py`

In [ ]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# 1) problema y datos: ¿irá esta persona al médico este año?
r = sm.datasets.randhie.load_pandas().data
y = (r.pop("mdvis") > 0).astype(int)
# 2) reservar la prueba antes de mirar nada más
X_e, X_p, y_e, y_p = train_test_split(r, y, test_size=0.25,
                                      random_state=0, stratify=y)
# 3) comparar candidatos solo con los datos de entrenamiento
candidatos = {
    "logística": make_pipeline(StandardScaler(),
                               LogisticRegression()),
    "potenciación": HistGradientBoostingClassifier(random_state=0)}
for nombre, m in candidatos.items():
    auc = cross_val_score(m, X_e, y_e, cv=5, scoring="roc_auc")
    print(f"{nombre:<13} AUC validación cruzada {auc.mean():.3f}"
          f" ± {auc.std():.3f}")
# 4) evaluar una sola vez el elegido en la prueba
final = candidatos["potenciación"].fit(X_e, y_e)
print("AUC en prueba:",
      round(roc_auc_score(y_p, final.predict_proba(X_p)[:, 1]), 3))
# 5) interpretar con el modelo sencillo
log = candidatos["logística"].fit(X_e, y_e)
coef = pd.Series(log[-1].coef_[0], X_e.columns).sort_values()
print("coeficientes estandarizados, extremos:")
print(coef.iloc[[0, 1, -2, -1]].round(2).to_string())

### ¿Ayudarían más datos?  ·  `d6_curva.py`

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import learning_curve, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

r = sm.datasets.randhie.load_pandas().data
y = (r.pop("mdvis") > 0).astype(int)
tam = [200, 1000, 4000, 16000]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for nombre, m in [
        ("logística", make_pipeline(StandardScaler(),
                                    LogisticRegression())),
        ("potenciación", HistGradientBoostingClassifier(
            random_state=0))]:
    n, ent, val = learning_curve(m, r, y, train_sizes=tam, cv=cv,
                                 scoring="roc_auc", shuffle=True,
                                 random_state=0)
    print(nombre)
    for k, e, v in zip(n, ent.mean(1), val.mean(1)):
        print(f"  {k:>6} casos: AUC entrenamiento {e:.3f},"
              f" validación {v:.3f}")

### Guardar y reutilizar un modelo  ·  `d5_guardar.py`

In [ ]:
import joblib
import statsmodels.api as sm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

r = sm.datasets.randhie.load_pandas().data
y = (r.pop("mdvis") > 0).astype(int)
modelo = make_pipeline(StandardScaler(), LogisticRegression())
modelo.fit(r, y)
joblib.dump(modelo, "modelo_rand.joblib")      # se guarda entero
cargado = joblib.load("modelo_rand.joblib")    # otro día
gratis = r.lncoins == 0                         # seguro sin copago
tipicos = r.groupby(gratis).median()           # persona mediana
tipicos.index = ["con copago", "gratuito"]
print(tipicos[["lncoins", "fmde", "disea"]].round(2))
print(cargado.predict_proba(tipicos)[:, 1].round(3))
print("mismas predicciones:",
      (cargado.predict(r) == modelo.predict(r)).all())

**Ejercicio.** Entrena el modelo con los años 1960-1979 y prueba con 1980-2009. ¿Sigue ganando la regla escrita a mano?

> Ball, L., Leigh, D. y Loungani, P. (2017). Okun's Law: Fit at 50? *Journal of Money, Credit and Banking*, 49(7), 1413–1441. https://doi.org/10.1111/jmcb.12420